# LangChain RAG Comparison — Rigorous Evaluation

**Goal**: Rigorously test our RagAnswerer template against plain RAG (LangChain-style).

Source: [LLM Powered Autonomous Agents](https://lilianweng.github.io/posts/2023-06-23-agent/) by Lilian Weng.

### Experiment Design

| Factor | Value |
|--------|-------|
| **Questions** | 8 (factual, analytical, multi-hop, synthesis, abstention) |
| **Conditions** | A. Plain RAG (LangChain-style), B. RagAnswerer |
| **Runs per condition** | 3 (to measure variance) |
| **Retrieved chunks** | k=4 per question |

### Metrics

| Metric | What it measures |
|--------|------------------|
| **Evidence markers** | Specific facts/terms from context that appear in output |
| **Citation count** | Explicit source references ("Source:", "Based on:", parenthetical refs) |
| **Faithfulness** | Numeric/factual claims grounded in context vs hallucinated |
| **Abstention** | Does the model correctly say "I don't know" when context is insufficient? |

---

```bash
pip install mycontext-ai litellm langchain langchain-text-splitters langchain-community langchain-openai bs4
```

In [ ]:
import os, time, re, json
from datetime import datetime
os.environ.setdefault('OPENAI_API_KEY', 'sk-...')
PROVIDER = 'openai'
NUM_RUNS = 3

## Step 1 — Load, Split, Index

In [ ]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

bs4_strainer = bs4.SoupStrainer(class_=('post-title', 'post-header', 'post-content'))
loader = WebBaseLoader(
    web_paths=('https://lilianweng.github.io/posts/2023-06-23-agent/',),
    bs_kwargs={'parse_only': bs4_strainer},
)
docs = loader.load()
print(f'Loaded {len(docs)} doc(s), {len(docs[0].page_content)} chars')

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, add_start_index=True)
all_splits = text_splitter.split_documents(docs)
print(f'Split into {len(all_splits)} chunks')

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(documents=all_splits)
print('Indexed')

## Step 2 — Question Battery

8 questions across 5 difficulty levels, each with question-specific evidence markers.

In [ ]:
QUESTIONS = [
    {
        'id': 'Q1', 'type': 'factual',
        'query': 'What is task decomposition?',
        'markers': [
            ('CoT / Chain of Thought', ['chain of thought', 'cot']),
            ('Tree of Thoughts', ['tree of thoughts']),
            ('LLM+P', ['llm+p']),
            ('PDDL', ['pddl']),
            ('subgoals', ['subgoal']),
            ('step by step', ['step by step']),
            ('human inputs', ['human input']),
            ('task-specific instructions', ['task-specific', 'task specific']),
        ],
    },
    {
        'id': 'Q2', 'type': 'factual',
        'query': 'What types of memory are discussed for autonomous agents?',
        'markers': [
            ('sensory memory', ['sensory memory']),
            ('short-term / working', ['short-term', 'working memory']),
            ('long-term memory', ['long-term memory']),
            ('explicit / declarative', ['explicit', 'declarative']),
            ('implicit / procedural', ['implicit', 'procedural']),
            ('vector store', ['vector store']),
            ('MIPS', ['mips', 'maximum inner product']),
            ('7 items / Miller', ['7 item', 'miller']),
        ],
    },
    {
        'id': 'Q3', 'type': 'analytical',
        'query': 'How does Chain of Thought (CoT) relate to Tree of Thoughts (ToT)? What is the key difference?',
        'markers': [
            ('CoT', ['chain of thought', 'cot']),
            ('ToT / Tree of Thoughts', ['tree of thoughts']),
            ('extends / builds on', ['extend', 'build']),
            ('multiple reasoning', ['multiple reasoning', 'multiple thought', 'multiple possibilities']),
            ('tree structure', ['tree structure']),
            ('BFS or DFS', ['bfs', 'dfs', 'breadth', 'depth']),
            ('classifier / majority vote', ['classifier', 'majority vote']),
            ('Wei et al 2022', ['wei', '2022']),
            ('Yao et al 2023', ['yao', '2023']),
        ],
    },
    {
        'id': 'Q4', 'type': 'analytical',
        'query': 'What is the ReAct framework and how does it combine reasoning with acting?',
        'markers': [
            ('ReAct', ['react']),
            ('Yao et al 2023', ['yao', '2023']),
            ('Thought/Action/Observation', ['thought', 'action', 'observation']),
            ('reasoning traces', ['reasoning trace']),
            ('action space', ['action space']),
            ('language space', ['language space']),
            ('interact with environment', ['interact', 'environment']),
            ('Wikipedia', ['wikipedia']),
        ],
    },
    {
        'id': 'Q5', 'type': 'multi-hop',
        'query': 'How do planning, memory, and tool use work together in an LLM-powered agent system?',
        'markers': [
            ('planning', ['planning']),
            ('memory', ['memory']),
            ('tool use', ['tool use', 'tool']),
            ('task decomposition', ['task decomposition', 'decompos']),
            ('self-reflection', ['self-reflection', 'self reflection', 'reflect']),
            ('external APIs', ['api', 'external']),
            ('vector store / retrieval', ['vector store', 'retriev']),
            ('subgoals', ['subgoal']),
        ],
    },
    {
        'id': 'Q6', 'type': 'multi-hop',
        'query': 'Compare the MIPS algorithms discussed in the blog. Which ones are mentioned and what are their key properties?',
        'markers': [
            ('FAISS', ['faiss']),
            ('ScaNN', ['scann']),
            ('HNSW', ['hnsw']),
            ('ANNOY', ['annoy']),
            ('LSH', ['lsh', 'locality-sensitive']),
            ('approximate nearest neighbors', ['approximate nearest', 'ann']),
            ('quantization', ['quantization']),
            ('hashing', ['hash']),
        ],
    },
    {
        'id': 'Q7', 'type': 'abstention',
        'query': 'What does the blog say about fine-tuning GPT-4 specifically for task decomposition?',
        'markers': [],
        'expect_abstention': True,
    },
    {
        'id': 'Q8', 'type': 'abstention',
        'query': 'According to the blog, what are the benchmark scores of Claude vs GPT-4 on agent tasks?',
        'markers': [],
        'expect_abstention': True,
    },
]

print(f'{len(QUESTIONS)} questions:')
for q in QUESTIONS:
    print(f'  {q["id"]} [{q["type"]}] {q["query"][:70]}...' if len(q['query']) > 70 else f'  {q["id"]} [{q["type"]}] {q["query"]}')

## Step 3 — Retrieve + Run (3 runs x 2 conditions x 8 questions)

In [ ]:
from mycontext.core import Context
from mycontext.foundation import Directive
from mycontext.templates.free.specialized import RagAnswerer
import litellm
litellm.drop_params = True

rag_template = RagAnswerer()
results = []

for run_idx in range(NUM_RUNS):
    print(f'\n--- Run {run_idx+1}/{NUM_RUNS} ---')
    for q in QUESTIONS:
        # Retrieve
        retrieved = vector_store.similarity_search(q['query'], k=4)
        context_text = '\n\n'.join(
            f'Source: {d.metadata}\nContent: {d.page_content}' for d in retrieved
        )

        for cond in ['A_plain_rag', 'B_rag_answerer']:
            if cond == 'A_plain_rag':
                prompt = (
                    'You are a helpful assistant. Use the following context in your response:\n\n'
                    f'{context_text}\n\nQuestion: {q["query"]}'
                )
                ctx = Context(directive=Directive(content=prompt))
            else:
                ctx = rag_template.build_context(
                    question=q['query'], retrieved_docs=context_text, task='answer'
                )

            t0 = time.time()
            res = ctx.execute(provider=PROVIDER)
            elapsed = time.time() - t0

            results.append({
                'run': run_idx, 'question_id': q['id'], 'question_type': q['type'],
                'condition': cond, 'output': res.response, 'time': elapsed,
                'context_chars': len(context_text),
            })
            print(f'  {q["id"]} {cond}: {elapsed:.1f}s ({len(res.response)} chars)')

print(f'\nTotal runs: {len(results)}')

## Step 4 — Scoring

In [ ]:
ABSTENTION_PHRASES = [
    'cannot find', 'not mention', 'does not mention', 'not discuss',
    'does not discuss', 'not contain', 'does not contain',
    'no information', 'not available', 'not address', 'does not address',
    'not enough information', 'insufficient', 'not in the', 'not covered',
    'does not cover', 'does not provide', 'not specifically', 'does not specifically',
    'i don\'t have', 'unable to find', 'not explicitly',
]

CITATION_PATTERNS = [
    r'\(source[:\s]',
    r'\(based on[:\s]',
    r'\(sources?[:\s]',
    r'source:\s',
    r'\[context\s*\d',
    r'\[source\s*\d',
    r'\(from\s',
    r'according to the (context|source|document|retrieved|blog|provided)',
]

def score_output(output, question):
    out_lower = output.lower()
    markers = question.get('markers', [])
    marker_hits = sum(1 for _, alts in markers if any(a.lower() in out_lower for a in alts))
    marker_total = len(markers)

    citation_count = sum(len(re.findall(p, out_lower)) for p in CITATION_PATTERNS)

    abstains = any(phrase in out_lower for phrase in ABSTENTION_PHRASES)
    expect_abstention = question.get('expect_abstention', False)
    abstention_correct = (abstains == expect_abstention)

    return {
        'marker_hits': marker_hits,
        'marker_total': marker_total,
        'marker_pct': marker_hits / marker_total if marker_total > 0 else None,
        'citation_count': citation_count,
        'abstains': abstains,
        'expect_abstention': expect_abstention,
        'abstention_correct': abstention_correct,
    }

q_lookup = {q['id']: q for q in QUESTIONS}
for r in results:
    r['scores'] = score_output(r['output'], q_lookup[r['question_id']])

print('Scoring complete.')

## Step 5 — Results Dashboard

In [ ]:
from collections import defaultdict

def avg(lst):
    return sum(lst) / len(lst) if lst else 0

# --- 5a: Per-question evidence markers (averaged across runs) ---
print('='*75)
print('5a. EVIDENCE MARKERS (avg across runs)')
print('='*75)
print(f'{"Q":<5} {"Type":<12} {"A_plain":>10} {"B_rag":>10} {"Max":>6} {"Winner":>10}')
print('-'*55)

q_scores = defaultdict(lambda: defaultdict(list))
for r in results:
    if r['scores']['marker_total'] and r['scores']['marker_total'] > 0:
        q_scores[r['question_id']][r['condition']].append(r['scores']['marker_hits'])

a_wins, b_wins, ties = 0, 0, 0
for q in QUESTIONS:
    if not q['markers']:
        continue
    a_avg = avg(q_scores[q['id']].get('A_plain_rag', [0]))
    b_avg = avg(q_scores[q['id']].get('B_rag_answerer', [0]))
    mx = len(q['markers'])
    if b_avg > a_avg:
        winner = 'B (Rag)'
        b_wins += 1
    elif a_avg > b_avg:
        winner = 'A (Plain)'
        a_wins += 1
    else:
        winner = 'TIE'
        ties += 1
    print(f'{q["id"]:<5} {q["type"]:<12} {a_avg:>10.1f} {b_avg:>10.1f} {mx:>6} {winner:>10}')

print(f'\nWins: A={a_wins}, B={b_wins}, Ties={ties}')

In [ ]:
# --- 5b: Citation counts ---
print('='*75)
print('5b. CITATION COUNTS (avg across all questions and runs)')
print('='*75)
cit_a = [r['scores']['citation_count'] for r in results if r['condition'] == 'A_plain_rag']
cit_b = [r['scores']['citation_count'] for r in results if r['condition'] == 'B_rag_answerer']
print(f'A. Plain RAG:   avg {avg(cit_a):.1f} citations/response (min={min(cit_a)}, max={max(cit_a)})')
print(f'B. RagAnswerer: avg {avg(cit_b):.1f} citations/response (min={min(cit_b)}, max={max(cit_b)})')

# Per question
print(f'\n{"Q":<5} {"A_plain":>10} {"B_rag":>10}')
print('-'*27)
for q in QUESTIONS:
    ca = avg([r['scores']['citation_count'] for r in results if r['question_id'] == q['id'] and r['condition'] == 'A_plain_rag'])
    cb = avg([r['scores']['citation_count'] for r in results if r['question_id'] == q['id'] and r['condition'] == 'B_rag_answerer'])
    print(f'{q["id"]:<5} {ca:>10.1f} {cb:>10.1f}')

In [ ]:
# --- 5c: Abstention accuracy (Q7, Q8) ---
print('='*75)
print('5c. ABSTENTION ACCURACY (should say "not in context" for Q7, Q8)')
print('='*75)
for q in QUESTIONS:
    if not q.get('expect_abstention'):
        continue
    for cond, label in [('A_plain_rag', 'A. Plain RAG'), ('B_rag_answerer', 'B. RagAnswerer')]:
        runs = [r for r in results if r['question_id'] == q['id'] and r['condition'] == cond]
        correct = sum(1 for r in runs if r['scores']['abstention_correct'])
        print(f'  {q["id"]} {label}: {correct}/{len(runs)} correct abstentions')
    print()

In [ ]:
# --- 5d: Latency ---
print('='*75)
print('5d. LATENCY (seconds, avg across runs)')
print('='*75)
time_a = [r['time'] for r in results if r['condition'] == 'A_plain_rag']
time_b = [r['time'] for r in results if r['condition'] == 'B_rag_answerer']
print(f'A. Plain RAG:   avg {avg(time_a):.1f}s')
print(f'B. RagAnswerer: avg {avg(time_b):.1f}s')

In [ ]:
# --- 5e: Variance (std dev of evidence markers across runs) ---
print('='*75)
print('5e. VARIANCE (std dev of evidence markers across 3 runs)')
print('='*75)
import statistics
print(f'{"Q":<5} {"A_std":>8} {"B_std":>8}')
print('-'*23)
for q in QUESTIONS:
    if not q['markers']:
        continue
    vals_a = q_scores[q['id']].get('A_plain_rag', [0])
    vals_b = q_scores[q['id']].get('B_rag_answerer', [0])
    std_a = statistics.stdev(vals_a) if len(vals_a) > 1 else 0
    std_b = statistics.stdev(vals_b) if len(vals_b) > 1 else 0
    print(f'{q["id"]:<5} {std_a:>8.2f} {std_b:>8.2f}')

## Step 6 — Aggregate Summary

In [ ]:
# Overall aggregation across all non-abstention questions
all_a_markers = [r['scores']['marker_hits'] for r in results if r['condition'] == 'A_plain_rag' and r['scores']['marker_total'] > 0]
all_b_markers = [r['scores']['marker_hits'] for r in results if r['condition'] == 'B_rag_answerer' and r['scores']['marker_total'] > 0]
all_a_pct = [r['scores']['marker_pct'] for r in results if r['condition'] == 'A_plain_rag' and r['scores']['marker_pct'] is not None]
all_b_pct = [r['scores']['marker_pct'] for r in results if r['condition'] == 'B_rag_answerer' and r['scores']['marker_pct'] is not None]

abs_a = [r for r in results if r['condition'] == 'A_plain_rag' and r['scores']['expect_abstention']]
abs_b = [r for r in results if r['condition'] == 'B_rag_answerer' and r['scores']['expect_abstention']]

print('='*75)
print('AGGREGATE SUMMARY')
print('='*75)
print(f'{"":<30} {"A. Plain RAG":>15} {"B. RagAnswerer":>15}')
print('-'*62)
print(f'{"Avg evidence markers":.<30} {avg(all_a_markers):>15.1f} {avg(all_b_markers):>15.1f}')
print(f'{"Avg marker % (recall)":.<30} {avg(all_a_pct):>14.1%} {avg(all_b_pct):>14.1%}')
print(f'{"Avg citations/response":.<30} {avg(cit_a):>15.1f} {avg(cit_b):>15.1f}')
print(f'{"Abstention accuracy":.<30} {sum(r["scores"]["abstention_correct"] for r in abs_a)}/{len(abs_a):>11} {sum(r["scores"]["abstention_correct"] for r in abs_b)}/{len(abs_b):>11}')
print(f'{"Avg latency (s)":.<30} {avg(time_a):>15.1f} {avg(time_b):>15.1f}')
print(f'{"Evidence marker wins":.<30} {a_wins:>15} {b_wins:>15}')

lift = ((avg(all_b_markers) / max(avg(all_a_markers), 0.01)) - 1) * 100
print(f'\nRagAnswerer evidence marker lift vs Plain RAG: {lift:+.1f}%')

## Step 7 — Read Sample Outputs

In [ ]:
# Show first run outputs side by side for each question
for q in QUESTIONS:
    print(f'\n{"#"*60}')
    print(f'{q["id"]} [{q["type"]}]: {q["query"]}')
    print('#'*60)
    for cond, label in [('A_plain_rag', 'A. Plain RAG'), ('B_rag_answerer', 'B. RagAnswerer')]:
        r = next(r for r in results if r['run'] == 0 and r['question_id'] == q['id'] and r['condition'] == cond)
        s = r['scores']
        meta = f'markers={s["marker_hits"]}/{s["marker_total"]}' if s['marker_total'] > 0 else 'abstention'
        meta += f', cites={s["citation_count"]}, abstains={s["abstains"]}'
        print(f'\n--- {label} ({meta}) ---')
        print(r['output'][:800] + ('...' if len(r['output']) > 800 else ''))

## Step 8 — Save Report

In [ ]:
report = {
    'experiment': 'LangChain RAG Comparison — Rigorous',
    'date': datetime.now().isoformat(),
    'source': 'lilianweng.github.io/posts/2023-06-23-agent/',
    'provider': PROVIDER,
    'num_runs': NUM_RUNS,
    'num_questions': len(QUESTIONS),
    'num_chunks': len(all_splits),
    'aggregate': {
        'A_plain_rag': {
            'avg_evidence_markers': round(avg(all_a_markers), 2),
            'avg_marker_pct': round(avg(all_a_pct), 4),
            'avg_citations': round(avg(cit_a), 2),
            'avg_latency': round(avg(time_a), 2),
            'abstention_accuracy': f'{sum(r["scores"]["abstention_correct"] for r in abs_a)}/{len(abs_a)}',
        },
        'B_rag_answerer': {
            'avg_evidence_markers': round(avg(all_b_markers), 2),
            'avg_marker_pct': round(avg(all_b_pct), 4),
            'avg_citations': round(avg(cit_b), 2),
            'avg_latency': round(avg(time_b), 2),
            'abstention_accuracy': f'{sum(r["scores"]["abstention_correct"] for r in abs_b)}/{len(abs_b)}',
        },
        'evidence_marker_wins': {'A': a_wins, 'B': b_wins, 'tie': ties},
        'lift_pct': round(lift, 2),
    },
    'per_question': [
        {
            'id': q['id'], 'type': q['type'], 'query': q['query'],
            'A_avg_markers': round(avg(q_scores[q['id']].get('A_plain_rag', [0])), 2) if q['markers'] else None,
            'B_avg_markers': round(avg(q_scores[q['id']].get('B_rag_answerer', [0])), 2) if q['markers'] else None,
            'max_markers': len(q['markers']) if q['markers'] else None,
        }
        for q in QUESTIONS
    ],
}

outpath = 'langchain_rag_comparison_results.json'
with open(outpath, 'w') as f:
    json.dump(report, f, indent=2)
print(f'Saved to {outpath}')